# The DPDP Act Compliance Filter: Parameter-Level Machine Unlearning
This notebook trains a LoRA adapter via gradient ascent and KL divergence to selectively erase PII.

In [ ]:
# Cell 1: pip install all requirements
!pip install torch transformers peft trl datasets rouge-score scipy accelerate bitsandbytes

In [ ]:
# Cell 2: clone GitHub repo or upload files
!git clone https://github.com/placeholder-repo/DPDP.git
%cd DPDP

In [ ]:
# Cell 3: run data_loader, load_model, frozen_reference
import config
from data.data_loader import load_and_prepare_datasets
from model.load_model import get_trainable_model
from model.frozen_reference import get_frozen_model

print("Loading datasets...")
forget_dataset, retain_dataset = load_and_prepare_datasets()

print("Loading training model with LoRA adapters...")
training_model = get_trainable_model()

print("Loading frozen baseline model for KL regularization...")
frozen_model = get_frozen_model()

In [ ]:
# Cell 4: run unlearn_trainer for all epochs
from training.unlearn_trainer import run_unlearning_loop

# Start the gradient ascent + KL divergence unlearning loop
training_model = run_unlearning_loop(training_model, frozen_model, forget_dataset, retain_dataset)

In [ ]:
# Cell 5: run ks_test and rouge_eval after each epoch 
# (Note: In a pure implementation, this would be injected inside the epoch loop, 
# but for Colab modularity we run it here as final validation)
from evaluation.ks_test import evaluate_forget_quality
from evaluation.rouge_eval import evaluate_model_utility
import json
import os

p_value = evaluate_forget_quality(training_model, frozen_model, forget_dataset)
rouge_score = evaluate_model_utility(training_model, retain_dataset)

# Log the metrics
os.makedirs('results', exist_ok=True)
with open('results/metrics_log.json', 'w') as f:
    json.dump({
        'forget_quality_pvalue': p_value,
        'rouge_l_score': rouge_score,
        'epochs_run': config.EPOCHS,
        'lambda_kl': config.LAMBDA_KL,
        'training_loss_final': 'Saved in console output'
    }, f, indent=4)

In [ ]:
# Cell 6: run adversarial_probe
from evaluation.adversarial_probe import run_adversarial_probe
run_adversarial_probe(training_model)

In [ ]:
# Cell 7: save LoRA adapter to ./adapters and zip for download
import os
from peft import PeftModel

output_dir = config.OUTPUT_DIR
os.makedirs(output_dir, exist_ok=True)

# Save only the adapter weights, keeping the footprint tiny.
training_model.save_pretrained(output_dir)
print(f"LoRA adapters saved successfully to {output_dir}")

!zip -r adapters.zip ./adapters
print("Zipped adapters. Ready for download.")